# Nanochat PPO and PPO-Standard Run on Kaggle

This notebook starts from the saved `d8_kaggle` SFT checkpoint and runs two Kaggle-friendly PPO variants:

- `scripts.chat_ppo`
- `scripts.chat_ppo_standard`

Recommended Kaggle setup:

- GPU enabled (`2x Tesla T4` expected)
- Internet enabled
- Attach `nanochat-codes`
- Attach `nanochat-d8-sft-cache`

The notebook imports only the tokenizer and `chatsft_checkpoints/d8_kaggle` from the SFT cache, then writes PPO checkpoints under `chatppo_checkpoints/d8_kaggle` and `chatppo_standard_checkpoints/d8_kaggle`.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time

INPUT_ROOT = Path('/kaggle/input')

CODE_INPUTS = sorted(INPUT_ROOT.glob('**/nanochat-codes'))
SFT_CACHE_INPUTS = sorted(INPUT_ROOT.glob('**/nanochat-d8-sft-cache'))

assert CODE_INPUTS, 'Attach the nanochat-codes Kaggle dataset'
assert SFT_CACHE_INPUTS, 'Attach the nanochat-d8-sft-cache Kaggle dataset'

CODE_INPUT = CODE_INPUTS[0]
SFT_CACHE_INPUT = SFT_CACHE_INPUTS[0]

WORK_REPO = Path('/kaggle/working/nanochat')
WORK_CACHE = Path('/kaggle/working/nanochat_cache')
HF_CACHE = Path('/kaggle/working/huggingface_cache')

for path in [WORK_REPO, WORK_CACHE, HF_CACHE]:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

shutil.copytree(CODE_INPUT, WORK_REPO, dirs_exist_ok=True)

required_cache_paths = [
    Path('tokenizer'),
    Path('chatsft_checkpoints') / 'd8_kaggle',
]
for rel_path in required_cache_paths:
    src_path = SFT_CACHE_INPUT / rel_path
    dst_path = WORK_CACHE / rel_path
    assert src_path.exists(), f'Missing required cache path in attached dataset: {src_path}'
    dst_path.parent.mkdir(parents=True, exist_ok=True)
    if src_path.is_dir():
        shutil.copytree(src_path, dst_path, dirs_exist_ok=True)
    else:
        shutil.copy2(src_path, dst_path)

os.environ['NANOCHAT_BASE_DIR'] = str(WORK_CACHE)
os.environ['HF_HOME'] = str(HF_CACHE)
os.environ['HF_DATASETS_CACHE'] = str(HF_CACHE / 'datasets')

print('Code input:', CODE_INPUT)
print('SFT cache input:', SFT_CACHE_INPUT)
print('Working repo:', WORK_REPO)
print('Nanochat cache:', WORK_CACHE)
print('HF cache:', HF_CACHE)
subprocess.run(['df', '-h', '/kaggle/working'], check=False)
subprocess.run(['du', '-sh', str(WORK_CACHE)], check=False)


## Install Dependencies


In [ ]:
packages = [
    'datasets>=4.0.0',
    'filelock>=3.0.0',
    'psutil>=7.1.0',
    'requests>=2.32.0',
    'tiktoken>=0.11.0',
    'tokenizers>=0.22.0',
    'transformers>=4.57.3',
    'wandb>=0.21.3',
    'zstandard>=0.25.0',
    'rustbpe>=0.1.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + packages)
print('Dependencies installed')


## Configure Runtime


In [ ]:
env = os.environ.copy()
env['NANOCHAT_BASE_DIR'] = str(WORK_CACHE)
env['HF_HOME'] = str(HF_CACHE)
env['HF_DATASETS_CACHE'] = str(HF_CACHE / 'datasets')
env['PYTHONUNBUFFERED'] = '1'

# PPO scripts do not use GradScaler, so float32 is safer on T4.
env['NANOCHAT_DTYPE'] = 'float32'
env['NANOCHAT_DISABLE_COMPILE'] = '1'
env['TORCH_COMPILE_DISABLE'] = '1'
env['NANOCHAT_ADAMW_ALLREDUCE'] = '1'
env['NANOCHAT_SERIAL_OPTIM_COMM'] = '1'
env['OMP_NUM_THREADS'] = '1'
env['NCCL_P2P_DISABLE'] = '1'
env['NCCL_IB_DISABLE'] = '1'
env['TORCH_NCCL_ASYNC_ERROR_HANDLING'] = '1'
env['NCCL_DEBUG'] = 'WARN'

os.environ.update(env)

import torch

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device count:', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f'GPU {i}:', torch.cuda.get_device_name(i))


## Validate Repo And SFT Cache


In [ ]:
assert (WORK_REPO / 'scripts' / 'chat_ppo.py').exists(), 'Missing scripts/chat_ppo.py'
assert (WORK_REPO / 'scripts' / 'chat_ppo_standard.py').exists(), 'Missing scripts/chat_ppo_standard.py'
assert (WORK_REPO / 'scripts' / 'chat_post_eval.py').exists(), 'Missing scripts/chat_post_eval.py'
assert (WORK_REPO / 'scripts' / 'chat_web.py').exists(), 'Missing scripts/chat_web.py'
assert (WORK_REPO / 'nanochat' / 'checkpoint_manager.py').exists(), 'Missing nanochat/checkpoint_manager.py'

MODEL_TAG = 'd8_kaggle'
sft_dir = WORK_CACHE / 'chatsft_checkpoints' / MODEL_TAG
tokenizer_dir = WORK_CACHE / 'tokenizer'
assert sft_dir.exists(), f'Missing SFT checkpoint dir: {sft_dir}'
assert tokenizer_dir.exists(), f'Missing tokenizer dir: {tokenizer_dir}'
assert sorted(sft_dir.glob('model_*.pt')), f'No model_*.pt files found in {sft_dir}'
assert sorted(sft_dir.glob('meta_*.json')), f'No meta_*.json files found in {sft_dir}'

subprocess.check_call(
    [
        sys.executable, '-m', 'py_compile',
        'scripts/chat_ppo.py',
        'scripts/chat_ppo_standard.py',
        'scripts/chat_post_eval.py',
        'scripts/chat_web.py',
        'nanochat/checkpoint_manager.py',
    ],
    cwd=WORK_REPO,
    env=env,
)

print('Setup complete')
print('SFT checkpoints:', [p.name for p in sorted(sft_dir.glob('model_*.pt'))[-5:]])
print('Tokenizer files:', sorted(p.name for p in tokenizer_dir.iterdir()))


## PPO Run


In [ ]:
NPROC = 2 if torch.cuda.is_available() and torch.cuda.device_count() >= 2 else 1
MODEL_TAG = 'd8_kaggle'
OVERWRITE_PPO_CHECKPOINTS = True

if OVERWRITE_PPO_CHECKPOINTS:
    shutil.rmtree(WORK_CACHE / 'chatppo_checkpoints' / MODEL_TAG, ignore_errors=True)
    shutil.rmtree(WORK_CACHE / 'chatppo_reward_checkpoints' / MODEL_TAG, ignore_errors=True)

ppo_args = [
    '-m', 'scripts.chat_ppo',

    '--run=dummy',

    '--policy-source=sft',
    f'--policy-tag={MODEL_TAG}',
    '--reference-source=sft',
    f'--reference-tag={MODEL_TAG}',

    '--preference-source=gsm8k',
    '--max-train-examples=4096',
    '--max-val-examples=512',

    '--rm-epochs=2',
    '--rm-batch-size=4',
    '--rm-lr=1e-4',
    '--rm-train-backbone=0',

    '--ppo-steps=300',
    '--ppo-epochs=1',
    '--prompt-batch-size=4',
    '--device-batch-size=2',
    '--max-new-tokens=256',

    '--temperature=1.0',
    '--top-k=0',

    '--clip-epsilon=0.2',
    '--kl-beta=0.02',
    '--kl-reduction=sum',
    '--advantage-whiten=1',

    '--embedding-lr=0.03',
    '--unembedding-lr=0.0008',
    '--matrix-lr=0.002',
    '--init-lr-frac=0.05',
    '--weight-decay=0.0',

    '--save-every=100',
]

if NPROC > 1:
    cmd = ['torchrun', '--standalone', f'--nproc_per_node={NPROC}'] + ppo_args
else:
    cmd = [sys.executable] + ppo_args

print('Running command:')
print(' '.join(cmd))
subprocess.run(cmd, cwd=WORK_REPO, env=env, check=True)

policy_dir = WORK_CACHE / 'chatppo_checkpoints' / MODEL_TAG
reward_dir = WORK_CACHE / 'chatppo_reward_checkpoints' / MODEL_TAG

print('PPO policy checkpoints:', [p.name for p in sorted(policy_dir.glob('model_*.pt'))])
print('PPO reward checkpoints:', [p.name for p in sorted(reward_dir.glob('reward_*.pt'))])


## PPO-Standard Run


In [ ]:
MODEL_TAG = 'd8_kaggle'
OVERWRITE_PPO_STANDARD_CHECKPOINTS = True

if OVERWRITE_PPO_STANDARD_CHECKPOINTS:
    shutil.rmtree(WORK_CACHE / 'chatppo_standard_checkpoints' / MODEL_TAG, ignore_errors=True)
    shutil.rmtree(WORK_CACHE / 'chatppo_standard_reward_checkpoints' / MODEL_TAG, ignore_errors=True)
    shutil.rmtree(WORK_CACHE / 'chatppo_standard_value_checkpoints' / MODEL_TAG, ignore_errors=True)

ppo_standard_env = env.copy()
ppo_standard_env['NANOCHAT_DTYPE'] = 'float32'

ppo_standard_args = [
    '-m', 'scripts.chat_ppo_standard',

    '--run=dummy',

    '--policy-source=sft',
    f'--policy-tag={MODEL_TAG}',
    '--reference-source=sft',
    f'--reference-tag={MODEL_TAG}',

    '--preference-source=gsm8k',
    '--max-train-examples=4096',
    '--max-val-examples=512',

    '--rm-epochs=2',
    '--rm-batch-size=4',
    '--rm-lr=1e-4',
    '--rm-train-backbone=0',

    '--ppo-steps=300',
    '--ppo-epochs=2',
    '--prompt-batch-size=2',
    '--device-batch-size=2',
    '--ppo-minibatch-size=2',
    '--max-new-tokens=256',

    '--temperature=1.0',
    '--top-k=0',

    '--clip-epsilon=0.2',
    '--value-clip-epsilon=0.2',
    '--kl-beta=0.02',
    '--gamma=1.0',
    '--gae-lambda=0.95',
    '--advantage-whiten=1',

    '--value-loss-coef=0.25',
    '--entropy-coef=0.0',
    '--value-lr=1e-4',
    '--value-train-backbone=1',

    '--embedding-lr=0.02',
    '--unembedding-lr=0.0005',
    '--matrix-lr=0.0015',
    '--init-lr-frac=0.05',
    '--weight-decay=0.0',

    '--save-every=100',
]

if NPROC > 1:
    cmd = ['torchrun', '--standalone', f'--nproc_per_node={NPROC}'] + ppo_standard_args
else:
    cmd = [sys.executable] + ppo_standard_args

print('Running command:')
print(' '.join(map(str, cmd)))
subprocess.run(list(map(str, cmd)), cwd=WORK_REPO, env=ppo_standard_env, check=True)

policy_dir = WORK_CACHE / 'chatppo_standard_checkpoints' / MODEL_TAG
reward_dir = WORK_CACHE / 'chatppo_standard_reward_checkpoints' / MODEL_TAG
value_dir = WORK_CACHE / 'chatppo_standard_value_checkpoints' / MODEL_TAG

print('PPO-standard policy checkpoints:', [p.name for p in sorted(policy_dir.glob('model_*.pt'))])
print('PPO-standard reward checkpoints:', [p.name for p in sorted(reward_dir.glob('reward_*.pt'))])
print('PPO-standard value checkpoints:', [p.name for p in sorted(value_dir.glob('value_*.pt'))])


## Inspect PPO Checkpoints


In [ ]:
families = [
    'chatppo_checkpoints',
    'chatppo_reward_checkpoints',
    'chatppo_standard_checkpoints',
    'chatppo_standard_reward_checkpoints',
    'chatppo_standard_value_checkpoints',
]
for family in families:
    checkpoint_dir = WORK_CACHE / family / MODEL_TAG
    print()
    print(family, checkpoint_dir)
    print('Exists:', checkpoint_dir.exists())
    if checkpoint_dir.exists():
        print('Files:', [p.name for p in sorted(checkpoint_dir.glob('*'))])

subprocess.run(['du', '-sh', str(WORK_CACHE)], check=False)


## Comparison Eval


In [ ]:
RUN_COMPARISON_EVAL = True
EVAL_MAX_PROBLEMS = 100

if RUN_COMPARISON_EVAL:
    models = ['sft=sft:d8_kaggle']

    ppo_dir = WORK_CACHE / 'chatppo_checkpoints'
    if (ppo_dir / MODEL_TAG).exists():
        models.append(f'ppo=@{ppo_dir}:{MODEL_TAG}')

    ppo_standard_dir = WORK_CACHE / 'chatppo_standard_checkpoints'
    if (ppo_standard_dir / MODEL_TAG).exists():
        models.append(f'ppo_standard=@{ppo_standard_dir}:{MODEL_TAG}')

    post_eval_args = [
        '-m', 'scripts.chat_post_eval',
    ]
    for model in models:
        post_eval_args.extend(['--models', model])
    post_eval_args.extend([
        '--max-problems', str(EVAL_MAX_PROBLEMS),
        '--batch-size', '2',
    ])

    if NPROC > 1:
        cmd = ['torchrun', '--standalone', f'--nproc_per_node={NPROC}'] + post_eval_args
    else:
        cmd = [sys.executable] + post_eval_args

    print('Running command:')
    print(' '.join(str(x) for x in cmd))
    subprocess.run([str(x) for x in cmd], cwd=WORK_REPO, env=env, check=True)
else:
    print('Skipping comparison eval')


## Inspect Saved Comparison Report


In [ ]:
report_path = WORK_CACHE / 'report' / 'chat-post-eval.md'
print(report_path)
print('Exists:', report_path.exists())
if report_path.exists():
    print(report_path.read_text())


## Output Manifest


In [ ]:
manifest = {
    'model_tag': MODEL_TAG,
    'sft_checkpoint_dir': str(WORK_CACHE / 'chatsft_checkpoints' / MODEL_TAG),
    'ppo_checkpoint_dir': str(WORK_CACHE / 'chatppo_checkpoints' / MODEL_TAG),
    'ppo_reward_checkpoint_dir': str(WORK_CACHE / 'chatppo_reward_checkpoints' / MODEL_TAG),
    'ppo_standard_checkpoint_dir': str(WORK_CACHE / 'chatppo_standard_checkpoints' / MODEL_TAG),
    'ppo_standard_reward_checkpoint_dir': str(WORK_CACHE / 'chatppo_standard_reward_checkpoints' / MODEL_TAG),
    'ppo_standard_value_checkpoint_dir': str(WORK_CACHE / 'chatppo_standard_value_checkpoints' / MODEL_TAG),
    'report': str(WORK_CACHE / 'report' / 'chat-post-eval.md'),
}
manifest_path = Path('/kaggle/working/nanochat_ppo_ppostandard_manifest.json')
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(manifest_path)
print(manifest_path.read_text())


In [ ]:
# Optional: save the PPO/PPO-standard checkpoint cache as a Kaggle Dataset.
import json

PPO_MODEL_TAG = MODEL_TAG
PPO_CACHE_DIR = Path('/kaggle/working/nanochat_d8_ppo_ppostandard_cache')

DATASET_ID = 'yshuaiqin/nanochat-d8-ppo-ppostandard-cache'
TITLE = 'nanochat d8 ppo ppostandard cache'
VERSION_MESSAGE = f'Save {PPO_MODEL_TAG} PPO and PPO-standard checkpoint cache'
UPLOAD_ACTION = 'create'  # use 'version' after the Dataset already exists

required_dirs = [
    WORK_CACHE / 'chatppo_checkpoints' / PPO_MODEL_TAG,
    WORK_CACHE / 'chatppo_reward_checkpoints' / PPO_MODEL_TAG,
    WORK_CACHE / 'chatppo_standard_checkpoints' / PPO_MODEL_TAG,
    WORK_CACHE / 'chatppo_standard_reward_checkpoints' / PPO_MODEL_TAG,
    WORK_CACHE / 'chatppo_standard_value_checkpoints' / PPO_MODEL_TAG,
    WORK_CACHE / 'tokenizer',
]
for required_dir in required_dirs:
    assert required_dir.exists(), f'Missing required directory: {required_dir}'

assert sorted((WORK_CACHE / 'chatppo_checkpoints' / PPO_MODEL_TAG).glob('model_*.pt')), 'No PPO policy checkpoints found'
assert sorted((WORK_CACHE / 'chatppo_standard_checkpoints' / PPO_MODEL_TAG).glob('model_*.pt')), 'No PPO-standard policy checkpoints found'

if PPO_CACHE_DIR.exists():
    shutil.rmtree(PPO_CACHE_DIR)
PPO_CACHE_DIR.mkdir(parents=True, exist_ok=True)

for family in [
    'chatppo_checkpoints',
    'chatppo_reward_checkpoints',
    'chatppo_standard_checkpoints',
    'chatppo_standard_reward_checkpoints',
    'chatppo_standard_value_checkpoints',
]:
    shutil.copytree(WORK_CACHE / family, PPO_CACHE_DIR / family)
shutil.copytree(WORK_CACHE / 'tokenizer', PPO_CACHE_DIR / 'tokenizer')

metadata = {
    'title': TITLE,
    'id': DATASET_ID,
    'licenses': [{'name': 'CC0-1.0'}],
}

metadata_path = PPO_CACHE_DIR / 'dataset-metadata.json'
metadata_path.write_text(json.dumps(metadata, indent=2))

print('Folder size:')
subprocess.run(['du', '-sh', str(PPO_CACHE_DIR)], check=False)

if UPLOAD_ACTION == 'create':
    cmd = [
        'kaggle', 'datasets', 'create',
        '-p', str(PPO_CACHE_DIR),
        '--dir-mode', 'tar',
    ]
elif UPLOAD_ACTION == 'version':
    cmd = [
        'kaggle', 'datasets', 'version',
        '-p', str(PPO_CACHE_DIR),
        '-m', VERSION_MESSAGE,
        '--dir-mode', 'tar',
    ]
else:
    raise ValueError("UPLOAD_ACTION must be 'create' or 'version'")

print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, text=True)

if result.returncode != 0:
    raise RuntimeError('Kaggle Dataset upload failed.')

print('Done.')
print(f'Dataset URL: https://www.kaggle.com/datasets/{DATASET_ID}')


## Serve the PPO/PPO-standard Chat Model

Run this after `chatppo_checkpoints/d8_kaggle` or `chatppo_standard_checkpoints/d8_kaggle` exists. Set `SERVE_SOURCE` to `ppo` or `ppo_standard` before running the first cell.


In [ ]:
import time
import requests

SERVE_SOURCE = 'ppo_standard'  # choose 'ppo' or 'ppo_standard'
SERVE_MODEL_TAG = MODEL_TAG
SERVER_PORT = 8000
BASE_URL = f'http://127.0.0.1:{SERVER_PORT}'

checkpoint_family = {
    'ppo': 'chatppo_checkpoints',
    'ppo_standard': 'chatppo_standard_checkpoints',
}[SERVE_SOURCE]
CHECKPOINT_DIR = WORK_CACHE / checkpoint_family

model_dir = CHECKPOINT_DIR / SERVE_MODEL_TAG
assert model_dir.exists(), f'Missing {SERVE_SOURCE} checkpoint directory: {model_dir}'
assert sorted(model_dir.glob('model_*.pt')), f'No model_*.pt files found in {model_dir}'
assert sorted(model_dir.glob('meta_*.json')), f'No meta_*.json files found in {model_dir}'

try:
    if server.poll() is None:
        print('Stopping existing NanoChat server...')
        server.terminate()
        server.wait(timeout=10)
        print('Stopped existing server.')
except NameError:
    pass
except Exception as exc:
    print('Could not stop existing server cleanly:', exc)
    try:
        server.kill()
        server.wait(timeout=10)
        print('Killed existing server.')
    except Exception:
        pass

messages = []

server_env = env.copy()
server_env['NANOCHAT_DISABLE_COMPILE'] = '1'
server_env['TORCH_COMPILE_DISABLE'] = '1'
server_env['OMP_NUM_THREADS'] = '1'

cmd = [
    sys.executable,
    '-m', 'scripts.chat_web',
    '--checkpoint-dir', str(CHECKPOINT_DIR),
    '--model-tag', SERVE_MODEL_TAG,
    '--num-gpus', '1',
    '--port', str(SERVER_PORT),
]

print('Starting NanoChat server:')
print(' '.join(cmd))
server = subprocess.Popen(cmd, cwd=WORK_REPO, env=server_env)
print(f'Server process started with PID {server.pid}.')

SERVER_READY = False
for _ in range(60):
    if server.poll() is not None:
        raise RuntimeError(f'NanoChat server exited early with code {server.returncode}')
    try:
        response = requests.get(f'{BASE_URL}/health', timeout=2)
        if response.ok:
            SERVER_READY = True
            break
    except requests.RequestException:
        pass
    time.sleep(2)

if SERVER_READY:
    print(f'NanoChat server is ready: {BASE_URL}')
else:
    print(f'NanoChat server is still loading. Wait a bit, then use: {BASE_URL}')


In [ ]:
import json
import requests

BASE = globals().get("BASE_URL", "http://127.0.0.1:8000")
messages = []

def ask(prompt, temperature=0.8, top_k=50, max_tokens=512):
    messages.append({"role": "user", "content": prompt})

    payload = {
        "messages": messages,
        "temperature": temperature,
        "top_k": top_k,
        "max_tokens": max_tokens,
    }

    print("Assistant:", end=" ", flush=True)
    answer = ""

    with requests.post(f"{BASE}/chat/completions", json=payload, stream=True, timeout=300) as response:
        response.raise_for_status()
        for line in response.iter_lines(decode_unicode=True):
            if not line or not line.startswith("data: "):
                continue

            data = json.loads(line[len("data: "):])
            if data.get("done"):
                break

            token = data.get("token", "")
            answer += token
            print(token, end="", flush=True)

    print()
    messages.append({"role": "assistant", "content": answer})
    return answer

print(f"Chatting with {BASE}. Type q, quit, or exit to stop. Type reset to clear history.")
while True:
    prompt = input("\nYou: ")
    command = prompt.strip().lower()
    if command in {"q", "quit", "exit"}:
        break
    if command == "reset":
        messages.clear()
        print("Chat history cleared.")
        continue
    ask(prompt)


In [ ]:
# Stop the NanoChat web server started by the serving cell.
import os
import time

SERVER_PORT = globals().get('SERVER_PORT', 8000)
stopped_any = False

server_process = globals().get('server')
if server_process is not None:
    if server_process.poll() is None:
        print(f'Stopping NanoChat server process {server_process.pid}...')
        server_process.terminate()
        try:
            server_process.wait(timeout=10)
            print('NanoChat server stopped.')
        except subprocess.TimeoutExpired:
            print('Server did not stop after terminate(); killing it...')
            server_process.kill()
            server_process.wait(timeout=10)
            print('NanoChat server killed.')
        stopped_any = True
    else:
        print(f'NanoChat server process already exited with code {server_process.returncode}.')
else:
    print('No notebook `server` variable found.')

try:
    import psutil

    current_pid = os.getpid()
    fallback_processes = []
    for proc in psutil.process_iter(['pid', 'name', 'cmdline']):
        if proc.info['pid'] == current_pid:
            continue
        cmdline = ' '.join(proc.info.get('cmdline') or [])
        if 'scripts.chat_web' not in cmdline:
            continue
        try:
            listening_on_port = any(
                conn.status == psutil.CONN_LISTEN
                and conn.laddr
                and conn.laddr.port == SERVER_PORT
                for conn in proc.net_connections(kind='inet')
            )
        except (psutil.AccessDenied, psutil.NoSuchProcess):
            listening_on_port = False
        if listening_on_port:
            fallback_processes.append(proc)

    for proc in fallback_processes:
        print(f'Stopping fallback chat_web process {proc.pid} on port {SERVER_PORT}...')
        proc.terminate()
    gone, alive = psutil.wait_procs(fallback_processes, timeout=10)
    for proc in alive:
        print(f'Killing fallback chat_web process {proc.pid}...')
        proc.kill()
    if fallback_processes:
        stopped_any = True
except Exception as exc:
    print('Fallback process scan skipped:', exc)

if stopped_any:
    time.sleep(2)
    print('Server shutdown complete.')
else:
    print('No running NanoChat server process found.')
